# Homework Starter — Stage 13: Productization

**This homework is self-contained.** It does not use your project data or your project
model — the cells below generate everything they need. Work through it in order.

You are building four things: a saved model, a Flask app that loads it at startup and
serves two routes, proof from this notebook that both routes work, and a README that
tells someone else how to call them.

In [19]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install scikit-learn
# !pip install joblib
# !pip install flask
# !pip install requests

## 1. Generate data and train a model

Nothing to fill in here — run it. Note `os.makedirs` **before** `joblib.dump`: without it
the save fails with `FileNotFoundError`, because `model/` does not exist yet.

In [20]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# the dataset for this homework - generated, not loaded
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

model = LinearRegression()
model.fit(X, y)

os.makedirs('model', exist_ok=True)          # BEFORE the dump, not after
joblib.dump(model, 'model/model.pkl')

# prove the file on disk is usable: load it back and predict with the loaded copy
reloaded = joblib.load('model/model.pkl')
print('saved to model/model.pkl')
print('prediction from the reloaded model:', reloaded.predict([[0.1, 0.2]])[0])

saved to model/model.pkl
prediction from the reloaded model: 23.58961171297328


## 2. Write `app.py`

Fill in the three TODOs, then run the cell — it writes `app.py` to disk.

**The model load stays where it is**, at the top of the file. It runs once when the app
starts. Do not move it inside a route: a route that loads the model on every request
re-reads the file from disk for every single caller.

In [21]:
app_code = '''
from flask import Flask, request, jsonify
import joblib

# loaded ONCE, at startup - not inside a route
model = joblib.load('model/model.pkl')
app = Flask(__name__)


@app.route('/predict', methods=['POST'])
def predict_post():
    data = request.get_json(silent=True) or {}
    features = data.get('features')
    # TODO 1: return jsonify({'error': ...}), 400 if features is missing
    #         or does not have exactly 2 values
    if features is None:
        return jsonify({'error': 'Missing "features" key in JSON body'}), 400
    if not isinstance(features, list) or len(features) != 2:
        return jsonify({'error': 'Features must be a list of exactly 2 numbers'}), 400
    try:
        features = [float(x) for x in features]
    except (ValueError, TypeError):
        return jsonify({'error': 'Features must be numbers'}), 400
    # TODO 2: otherwise predict and return jsonify({'prediction': <the number>})
    #         remember model.predict takes a LIST of rows: model.predict([features])
    prediction = model.predict([features])[0]
    return jsonify({'prediction': prediction})


@app.route('/predict/<f1>/<f2>', methods=['GET'])
def predict_get(f1, f2):
    # TODO 3: f1 and f2 arrive as STRINGS. Convert them to float, and return
    #         jsonify({'error': ...}), 400 if either one is not a number.
    #         Then predict with the same `model` above and return the same JSON
    #         shape as /predict does.
    try:
        f1_float = float(f1)
        f2_float = float(f2)
    except ValueError:
        return jsonify({'error': 'Path parameters must be numbers'}), 400
    
    prediction = model.predict([[f1_float, f2_float]])[0]
    return jsonify({'prediction': prediction})


if __name__ == '__main__':
    app.run(port=5000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('wrote app.py')

wrote app.py


## 3. Launch the server

This opens a **separate terminal window** and starts Flask there. Leave it running.
Every time you change `app.py`, close that window and run this cell again.

In [22]:
os.system("start cmd /k python app.py")
print('Flask launched in an external terminal window. Give it a few seconds to start.')

Flask launched in an external terminal window. Give it a few seconds to start.


## 4. Call your own API

Three calls: the POST route, the GET route, and one deliberately bad call. **Leave this
output visible in the notebook you submit — it is your testing evidence.**

In [25]:
import requests

BASE = 'http://127.0.0.1:5000'

try:
    r1 = requests.post(BASE + '/predict', json={'features': [0.1, 0.2]}, timeout=5)
    print('POST /predict          ', r1.status_code, r1.text.strip())

    r2 = requests.get(BASE + '/predict/0.1/0.2', timeout=5)
    print('GET  /predict/0.1/0.2  ', r2.status_code, r2.text.strip())

    # deliberately bad: not a number. This must be a 400 and a JSON error,
    # not a traceback in the server window.
    r3 = requests.get(BASE + '/predict/abc/0.2', timeout=5)
    print('GET  /predict/abc/0.2  ', r3.status_code, r3.text.strip())
except requests.exceptions.ConnectionError:
    print('No server on port 5000. Run the launch cell above, wait a few seconds,')
    print('then run this cell again.')

POST /predict           200 {"prediction":23.58961171297328}
GET  /predict/0.1/0.2   200 {"prediction":23.58961171297328}
GET  /predict/abc/0.2   400 {"error":"Path parameters must be numbers"}


## 5. Write `README.md`

Run the cell to get a template, then **edit the file** — replace every `TODO` with the
real thing, and paste in the responses you actually got above.

In [26]:
readme = '''# Stage 13 Homework - Prediction API

TODO: two sentences on what the model does.

## Running it

    python app.py

The server starts on http://127.0.0.1:5000 and loads model/model.pkl at startup.

## POST /predict

    curl -X POST http://127.0.0.1:5000/predict \\
         -H "Content-Type: application/json" \\
         -d "{\\"features\\": [0.1, 0.2]}"

Response: 200 {"prediction":23.58961171297328}

## GET /predict/<f1>/<f2>

    curl http://127.0.0.1:5000/predict/0.1/0.2

Response: 200 {"prediction":23.58961171297328}

## Bad input
400 {"error":"Path parameters must be numbers"}
'''

with open('README.md', 'w') as f:
    f.write(readme)
print('wrote README.md - now edit it')

wrote README.md - now edit it


### Save Notebook
Remember to save as `homework13_productization_submission.ipynb`.